In [ ]:
from collections import deque
import copy

In [ ]:
ROW_IDS = range(9)
COL_IDS = range(9)

# Variables:(row, col) pairs
CELL_POSITIONS = [(row_id, col_id) for row_id in ROW_IDS for col_id in COL_IDS]


def compute_subgrid_id(row_id, col_id):
    return row_id // 3, col_id // 3


PEER_MAP = {coord: set() for coord in CELL_POSITIONS}
for row_id in ROW_IDS:
    for col_id in COL_IDS:
        current_cell = (row_id, col_id)

        # For Same row
        for alt_col in COL_IDS:
            if alt_col != col_id:
                PEER_MAP[current_cell].add((row_id, alt_col))

        # For Same column
        for alt_row in ROW_IDS:
            if alt_row != row_id:
                PEER_MAP[current_cell].add((alt_row, col_id))

        # For Same 3x3 block
        block_row, block_col = compute_subgrid_id(row_id, col_id)
        for block_r in range(block_row * 3, block_row * 3 + 3):
            for block_c in range(block_col * 3, block_col * 3 + 3):
                if (block_r, block_c) != current_cell:
                    PEER_MAP[current_cell].add((block_r, block_c))

In [ ]:
def establish_domains(grid):
    candidate_map = {spot: set(range(1, 10)) for spot in CELL_POSITIONS}

    # Setting domains for given cells
    for row_id in ROW_IDS:
        for col_id in COL_IDS:
            preset = grid[row_id][col_id]
            if preset != 0:
                candidate_map[(row_id, col_id)] = {preset}

    # Pruning neighbors using the givens
    for row_id in ROW_IDS:
        for col_id in COL_IDS:
            preset = grid[row_id][col_id]
            if preset != 0:
                for peer in PEER_MAP[(row_id, col_id)]:
                    if preset in candidate_map[peer] and len(candidate_map[peer]) > 1:
                        candidate_map[peer].discard(preset)

    return candidate_map

In [ ]:
def assignment_ok(candidate_digit, cell, partial_assignment):
    for peer in PEER_MAP[cell]:
        if peer in partial_assignment and partial_assignment[peer] == candidate_digit:
            return False
    return True

In [ ]:
# Heuristics: MRV + Degree

def pick_unfilled_cell(candidate_map, partial_assignment):
    pending_cells = [coord for coord in CELL_POSITIONS if coord not in partial_assignment]

    # MRV: minimum domain size
    min_width = min(len(candidate_map[coord]) for coord in pending_cells)
    best_options = [coord for coord in pending_cells if len(candidate_map[coord]) == min_width]

    if len(best_options) == 1:
        return best_options[0]

    # Degree heuristic: most constraints on other unassigned vars
    preferred_cell = None
    highest_degree = -1
    for coord in best_options:
        current_degree = sum(1 for peer in PEER_MAP[coord] if peer not in partial_assignment)
        if current_degree > highest_degree:
            highest_degree = current_degree
            preferred_cell = coord
    return preferred_cell

In [ ]:
# Heuristic: LCV (Least Constraining Value)

def rank_candidate_digits(cell, candidate_map):
    scoring = []
    for digit in candidate_map[cell]:
        # Count how many neighbor-domain values would be removed
        peer_conflicts = 0
        for peer in PEER_MAP[cell]:
            if digit in candidate_map[peer]:
                peer_conflicts += 1
        scoring.append((peer_conflicts, digit))

    # Least constraining first (smallest removed_count)
    scoring.sort()
    return [digit for (_, digit) in scoring]

In [ ]:
# Forward checking

def apply_forward_check(cell, digit, candidate_map):
    next_candidate_map = copy.deepcopy(candidate_map)
    next_candidate_map[cell] = {digit}

    for peer in PEER_MAP[cell]:
        if digit in next_candidate_map[peer]:
            # If neighbor already forced to this value, we must not remove its only domain value
            if len(next_candidate_map[peer]) == 1:
                return False, None
            next_candidate_map[peer].remove(digit)
            if len(next_candidate_map[peer]) == 0:
                return False, None

    return True, next_candidate_map

In [ ]:
# AC-3 (Arc Consistency)

def enforce_ac3(candidate_map):
    propagation_queue = deque()
    for origin in CELL_POSITIONS:
        for target in PEER_MAP[origin]:
            propagation_queue.append((origin, target))

    working_map = {coord: set(values) for coord, values in candidate_map.items()}

    def revise(origin_cell, neighbor_cell):
        trimmed = False
        to_prune = set()

        # For each x in domain(origin_cell), check if there is a y in domain(neighbor_cell) such that x != y.
        for origin_digit in working_map[origin_cell]:
            if not any(origin_digit != neighbor_digit for neighbor_digit in working_map[neighbor_cell]):
                # origin_digit has no supporting value in neighbor_cell
                to_prune.add(origin_digit)

        if to_prune:
            working_map[origin_cell] -= to_prune
            trimmed = True

        return trimmed

    while propagation_queue:
        origin_cell, neighbor_cell = propagation_queue.popleft()
        if revise(origin_cell, neighbor_cell):
            if len(working_map[origin_cell]) == 0:
                return False, candidate_map  # failure
            for another_neighbor in PEER_MAP[origin_cell]:
                if another_neighbor != neighbor_cell:
                    propagation_queue.append((another_neighbor, origin_cell))

    return True, working_map

In [ ]:
# Backtracking search

def search_with_backtracking(partial_assignment, candidate_map, use_forward_check=True, use_ac3_in_search=False):
    # All variables assigned => solved
    if len(partial_assignment) == 81:
        return partial_assignment, candidate_map

    # Select variable with MRV + Degree
    chosen_cell = pick_unfilled_cell(candidate_map, partial_assignment)

    # Order values with LCV
    for candidate_digit in rank_candidate_digits(chosen_cell, candidate_map):

        if assignment_ok(candidate_digit, chosen_cell, partial_assignment):
            # Try assign
            partial_assignment[chosen_cell] = candidate_digit

            # Apply forward checking if enabled
            if use_forward_check:
                fc_ok, narrowed_map = apply_forward_check(chosen_cell, candidate_digit, candidate_map)
                if not fc_ok:
                    del partial_assignment[chosen_cell]
                    continue
                active_map = narrowed_map
            else:
                active_map = candidate_map

            # Optionally run AC-3 after each assignment
            if use_ac3_in_search:
                ac3_ok, refined_map = enforce_ac3(active_map)
                if not ac3_ok:
                    del partial_assignment[chosen_cell]
                    continue
                downstream_map = refined_map
            else:
                downstream_map = active_map

            # Recurse
            result_assignment, final_candidates = search_with_backtracking(
                partial_assignment,
                downstream_map,
                use_forward_check,
                use_ac3_in_search,
            )
            if result_assignment is not None:
                return result_assignment, final_candidates

            # Backtrack
            del partial_assignment[chosen_cell]

    return None, candidate_map

In [ ]:
# Top-level Sudoku solver

def resolve_sudoku(grid, use_forward_check=True, use_ac3_pre=True, use_ac3_in_search=False):
    candidate_map = establish_domains(grid)

    # Initial assignment from given digits
    partial_assignment = {}
    for row_id in ROW_IDS:
        for col_id in COL_IDS:
            if grid[row_id][col_id] != 0:
                partial_assignment[(row_id, col_id)] = grid[row_id][col_id]

    # Run AC-3 once before search (typical)
    if use_ac3_pre:
        ac3_ready, candidate_map = enforce_ac3(candidate_map)
        if not ac3_ready:
            return None  # inconsistent puzzle

    # Backtracking search
    final_assignment, candidate_map = search_with_backtracking(
        partial_assignment,
        candidate_map,
        use_forward_check=use_forward_check,
        use_ac3_in_search=use_ac3_in_search,
    )

    if final_assignment is None:
        return None

    # Build solved board
    solved_grid = [[0] * 9 for _ in range(9)]
    for (row_id, col_id), digit in final_assignment.items():
        solved_grid[row_id][col_id] = digit

    return solved_grid

In [ ]:
# print func

def display_grid(grid):
    for row_index in range(9):
        line = ""
        for col_index in range(9):
            entry = grid[row_index][col_index]
            line += (str(entry) if entry != 0 else "_") + " "
            if col_index % 3 == 2 and col_index != 8:
                line += "| "
        print(line)
        if row_index % 3 == 2 and row_index != 8:
            print("-" * 21)

In [ ]:
# Example

if __name__ == "__main__":
    initial_grid = [
        [0, 0, 0, 2, 6, 0, 7, 0, 1],
        [6, 8, 0, 0, 7, 0, 0, 9, 0],
        [1, 9, 0, 0, 0, 4, 5, 0, 0],
        [8, 2, 0, 1, 0, 0, 0, 4, 0],
        [0, 0, 4, 6, 0, 2, 9, 0, 0],
        [0, 5, 0, 0, 0, 3, 0, 2, 8],
        [0, 0, 9, 3, 0, 0, 0, 7, 4],
        [0, 4, 0, 0, 5, 0, 0, 3, 6],
        [7, 0, 3, 0, 1, 8, 0, 0, 0],
    ]

    print("Initial puzzle:")
    display_grid(initial_grid)

    solved_grid = resolve_sudoku(
        initial_grid,
        use_forward_check=True,
        use_ac3_pre=True,
        use_ac3_in_search=True,  # set to False if you want simpler FC+MRV+LCV
    )

    print("\nSolved puzzle:")
    display_grid(solved_grid)
